In [0]:
%sql
CREATE DATABASE IF NOT EXISTS olist;

USE olist;

In [0]:
#Bibliotecas
import requests
import io
import pandas as pd

In [0]:
tabelas_olist = [
    "customers",
    "geolocation",
    "order_items",
    "order_payments",
    "order_reviews",
    "orders",
    "products",
    "sellers",
    "product_category_name_translation"
]

# Base URL dos arquivos Gits
base_url = "https://raw.githubusercontent.com/HudsonSanto/fabric-medallion-architecture/main"

#Loop para processamento dos CSVs da Olist
for tabela in tabelas_olist:
    print(f"Processando a tabela: {tabela}...")
    
    # Montagem limpa das URLs usando f-string sem duplicar barras
    if tabela == "product_category_name_translation":
        url_completa = f"{base_url}/{tabela}.csv"
    else:
        url_completa = f"{base_url}/olist_{tabela}_dataset.csv"
    
    try:
        response = requests.get(url_completa)
        
        if response.status_code != 200:
            print(f"Erro 404: Arquivo não encontrado em {url_completa}")
            continue
            
        # Tratamento com skip para o que tiver ruim
        pdf = pd.read_csv(io.StringIO(response.text), on_bad_lines='skip')
            
        if pdf.empty:
            print(f"Aviso: DataFrame vazio para {tabela}")
            continue
            
        df_spark = spark.createDataFrame(pdf)
        nome_tabela_bronze = f"lnd_bronze_{tabela}"
        
        df_spark.write \
            .format("delta") \
            .mode("overwrite") \
            .option("overwriteSchema", "true") \
            .saveAsTable(nome_tabela_bronze)
            
        print(f"Sucesso: {nome_tabela_bronze} gravada com {df_spark.count()} linhas!")
        
    except Exception as e:
        print(f"Erro ao processar {tabela}: {str(e)}")

print("\n--- CAMADA BRONZE 100% CONCLUÍDA ---")

Processando a tabela: customers...
Sucesso: lnd_bronze_customers gravada com 99441 linhas!
Processando a tabela: geolocation...
Sucesso: lnd_bronze_geolocation gravada com 1000163 linhas!
Processando a tabela: order_items...
Sucesso: lnd_bronze_order_items gravada com 112650 linhas!
Processando a tabela: order_payments...
Sucesso: lnd_bronze_order_payments gravada com 103886 linhas!
Processando a tabela: order_reviews...
Sucesso: lnd_bronze_order_reviews gravada com 90072 linhas!
Processando a tabela: orders...
Sucesso: lnd_bronze_orders gravada com 99441 linhas!
Processando a tabela: products...
Sucesso: lnd_bronze_products gravada com 32951 linhas!
Processando a tabela: sellers...
Sucesso: lnd_bronze_sellers gravada com 3095 linhas!
Processando a tabela: product_category_name_translation...
Sucesso: lnd_bronze_product_category_name_translation gravada com 71 linhas!

--- CAMADA BRONZE 100% CONCLUÍDA ---


In [0]:
%sql
SELECT * FROM lnd_bronze_customers LIMIT 5;

customer_id,customer_unique_id,customer_zip_code_prefix,customer_city,customer_state
c4bd3aefb216827f4698bd9d2b166a2f,5bf4ea2d98005b960eea0dbf652ef4e7,37440,caxambu,MG
4478451cfd0f1879300c03bcc3c5d023,81d40912b580322404315c09bfe10533,25520,sao joao de meriti,RJ
85ea40a9684026fe82ea78356c8d1eaa,4b293b0eba86e10fcecd40a7aac6608b,89885,sao carlos,SC
ffee99041e111172de2006fe9e90202a,5d7b01c0a09b912d9874a3fffdb858d5,65919,imperatriz,MA
353cfaa362cd0b203605de27ae42cca8,6811df2808205f25e24a7f84328e99f7,63660,taua,CE


In [0]:
%sql
DESCRIBE EXTENDED olist.bronze_orders;